# UR5e MuJoCo 시뮬레이션과 teleop

`ri_motion_v5_py312` 커널을 선택하세요. 마지막 셀에서 관절 슬라이더, Quest 3, SpaceMouse 중 하나로 가상 로봇을 조작합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
for repo_root in (cwd, *cwd.parents):
    if (repo_root / 'vai_robotics' / 'ur5e' / 'sim.py').is_file():
        break
else:
    raise FileNotFoundError('VAI_AIWORKER 저장소 안에서 노트북을 열어 주세요.')

simulator = [sys.executable, '-m', 'vai_robotics.ur5e.sim']

### 모델과 관절 동작 확인

뷰어를 열기 전에 UR5e 모델을 불러와 작은 관절 명령을 적용하고 결과를 확인합니다.

In [ ]:
check = subprocess.run([*simulator, '--check'], cwd=repo_root, capture_output=True, text=True)
print(check.stdout, end='')
if check.stderr:
    print(check.stderr, end='', file=sys.stderr)
check.check_returncode()

### 조작 방식 선택

`DEVICE`를 `sliders`, `quest`, `spacemouse` 중 하나로 설정하세요. Quest를 쓸 때는 `QUEST_HOST`에 헤드셋에서 접속할 PC의 LAN 주소를 입력하세요. SpaceMouse는 USB 수신기나 케이블로 PC에 연결합니다. 장치별 설정은 [UR5e 실행 안내](../README.md)를 참고하세요.

Quest는 선택한 손의 grip, SpaceMouse는 0번 버튼을 한 번 놓았다가 누른 채 조작합니다. 창을 닫거나 커널의 실행 중지 버튼을 누르면 세션이 끝납니다.

In [ ]:
import signal

DEVICE = 'sliders'
QUEST_HOST = ''
HAND = 'right'

if DEVICE == 'sliders':
    command = simulator
else:
    command = [sys.executable, '-u', '-m', 'vai_robotics.ur5e.teleop',
               '--backend', 'sim', '--input', DEVICE]
    if DEVICE == 'quest':
        if not QUEST_HOST:
            raise ValueError('QUEST_HOST에 PC의 LAN 주소를 입력하세요.')
        command += ['--quest-host', QUEST_HOST, '--hand', HAND]

with subprocess.Popen(command, cwd=repo_root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'시뮬레이터 종료 코드 {process.returncode}')
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            process.wait()